# 7. (3.25 points)

Given the following information for a company as of December 31, 2018:

In [1]:
import chainladder as cl

tri = cl.Triangle(
    data={
        "origin": [2013, 2013, 2013, 2013, 2013, 2013, 2014, 2014, 2014, 2014, 2014, 2015, 2015, 2015, 2015, 2016, 2016, 2016, 2017, 2017, 2018],
        "development": [2013, 2014, 2015, 2016, 2017, 2018, 2014, 2015, 2016, 2017, 2018, 2015, 2016, 2017, 2018, 2016, 2017, 2018, 2017, 2018, 2018],
        "Case Incurred Losses": [3384, 4282, 4813, 5408, 5394, 5362, 3499, 4438, 5646, 5532, 5512, 3483, 5566, 5539, 5398, 6502, 6197, 6222, 6656, 6658, 6595]
    },
    origin="origin",
    development="development",
    columns="Case Incurred Losses",
    cumulative=True
)

print(tri)
print(tri.link_ratio)

          12      24      36      48      60      72
2013  3384.0  4282.0  4813.0  5408.0  5394.0  5362.0
2014  3499.0  4438.0  5646.0  5532.0  5512.0     NaN
2015  3483.0  5566.0  5539.0  5398.0     NaN     NaN
2016  6502.0  6197.0  6222.0     NaN     NaN     NaN
2017  6656.0  6658.0     NaN     NaN     NaN     NaN
2018  6595.0     NaN     NaN     NaN     NaN     NaN
         12-24     24-36     36-48     48-60     60-72
2013  1.265366  1.124007  1.123624  0.997411  0.994067
2014  1.268362  1.272195  0.979809  0.996385       NaN
2015  1.598048  0.995149  0.974544       NaN       NaN
2016  0.953091  1.004034       NaN       NaN       NaN
2017  1.000300       NaN       NaN       NaN       NaN


* a. (1.75 points) Test the null-hypothesis that there are no calendar year influences in the data using Mack's methodology assuming an error probability of 5% where:
    * $E[Z] = 3.25$
    * $\text{Var}(Z) = 1.06$
    * The 97.5th percentile of the Standard Normal distribution is 1.96
* b. (1.5 points) The graph below shows the weighted residuals of case incurred losses for maturities of 24, 36, and 48 months.

  ![graph](2019_7.png)

Discuss whether the results above could be explained by each of the following:

* i. Sustained increase in calendar year rend rate in 2016.
* ii. Increase in case reserve adequacy in 2016.
* iii. Acceleration of loss payment in 2016.

## Solution to a. (Mack 1994, Appendix H: test for calendar year effects)

1. Rank each column of link ratios $F_k$. Ratios below the median are **S**, above are **L**; the median itself (odd count) is eliminated.
2. For each diagonal (calendar year) count $S_j$, $L_j$ and take $Z_j=\min(S_j,L_j)$. Diagonals with $S_j+L_j\le1$ are left out.
3. $Z=\sum Z_j$. Reject "no calendar year effects" (5% error) if $Z\notin E[Z]\pm1.96\sqrt{Var(Z)}$.

In [2]:
import numpy as np
import pandas as pd

lr = tri.link_ratio.to_frame()                     # origin x link column
A = lr.values
rows = []
cal = {}
for i in range(A.shape[0]):
    for k in range(A.shape[1]):
        if np.isnan(A[i, k]):
            continue
        col = A[:, k][~np.isnan(A[:, k])]
        med = np.median(col)
        label = "S" if A[i, k] < med else "L" if A[i, k] > med else "*"
        cy = 2013 + i + k + 1
        cal.setdefault(cy, []).append(label)

table = pd.DataFrame(
    [{"CY": cy, "S": v.count("S"), "L": v.count("L")} for cy, v in sorted(cal.items())]
).set_index("CY")
table["Z_j"] = table[["S", "L"]].min(axis=1)
print(table)

Z = table.loc[table["S"] + table["L"] >= 2, "Z_j"].sum()
EZ, VarZ = 3.25, 1.06
lo, hi = EZ - 1.96 * np.sqrt(VarZ), EZ + 1.96 * np.sqrt(VarZ)
print(f"Z = {Z},  95% range = ({lo:.3f}, {hi:.3f}),  reject = {not (lo <= Z <= hi)}")
assert Z == 1 and Z < lo

      S  L  Z_j
CY             
2014  0  0    0
2015  0  2    0
2016  0  3    0
2017  2  1    1
2018  4  0    0
Z = 1,  95% range = (1.232, 5.268),  reject = True


**a.** The S/L counts by calendar year are: 2015 (S=0, L=2), 2016 (0, 3), 2017 (2, 1), 2018 (4, 0). The 2014 diagonal has one ratio, so it is eliminated. $Z=0+0+1+0=1$.
The 95% range is $3.25\pm1.96\sqrt{1.06}=(1.23,\ 5.27)$. Since $Z=1<1.23$, **reject** the null hypothesis: there is evidence of calendar year influences in the data.
(The given $E[Z]=3.25$ and $Var(Z)=1.06$ match the sums of the per-diagonal values with $n=2,3,3,4$.)

## Solution to b.

The residuals are the weighted residuals $(f_{ik}-\hat f_k)\sqrt{C_{ik}}$. Recomputing them from the triangle reproduces the graph's pattern: calendar year 2016 is strongly positive at all three maturities (about +26, +12, +7), and 2017 and 2018 are negative at all three (about −3 to −16). (My recomputed values are close to the graph but not identical for 2017-2018, so I only rely on the pattern.)

In [3]:
C = tri.to_frame().values
recs = []
for k in range(3):                                   # link ratios 12-24, 24-36, 36-48
    rows_k = [i for i in range(C.shape[0]) if not np.isnan(C[i, k + 1])]
    fk = np.nansum(C[rows_k, k + 1]) / C[rows_k, k].sum()
    for i in rows_k:
        recs.append({"CY": 2013 + i + k + 1, "maturity": 12 * (k + 2),
                     "residual": (C[i, k + 1] / C[i, k] - fk) * np.sqrt(C[i, k])})
resid = pd.DataFrame(recs)
print(resid.pivot_table(index="CY", columns="maturity", values="residual").round(1))

maturity    24    36   48
CY                       
2014       6.5   NaN  NaN
2015       6.8   2.6  NaN
2016      26.2  12.5  7.1
2017     -16.2  -6.7 -3.1
2018     -12.5  -6.4 -3.5


**i. Sustained increase in the calendar year trend rate in 2016: no.** A sustained higher trend would push development above the pattern in 2016 *and in every later calendar year* (positive residuals continuing). The graph shows large positive residuals only in 2016 and negative ones in 2017 and 2018.

**ii. Increase in case reserve adequacy in 2016: yes.** Strengthening case reserves in 2016 raises incurred losses at every maturity in that one calendar year (all 2016 residuals positive). From then on case reserves are at the higher level, so later development is lower than the historical pattern, which has the strengthening in it (negative residuals in 2017 and 2018). This one-time jump followed by below-average development fits the graph.

**iii. Acceleration of loss payment in 2016: no (for incurred).** Case incurred = paid + case reserves. Paying claims faster moves dollars from case reserves to paid and leaves incurred unchanged, so incurred residuals would not show it. It would show up in a paid triangle (high paid development in 2016, then low afterwards), not in this case incurred data.

## chainladder-python gaps

- **Bug in `cl.ValuationCorrelation(total=True)`:** `z`, `z_expectation` and `z_variance` are right here (1.0, 3.25, 1.0625, matching the given values), but `z_critical` is `False`, which gives the wrong conclusion. The code builds the range from the **sum of the variances** (`range = (VarZ + ci2, VarZ - ci2)`) and compares `VarZ.sum()` with it, instead of using $E[Z]\pm1.96\sqrt{Var(Z)}$ and comparing $Z$. The correct answer is reject (1 < 1.23). The docstring's RAA example (Z=14, E=12.9) hides it because both versions say "not rejected".
- `total=True` returns only totals; there is no per-diagonal $S_j$/$L_j$/$Z_j$ table like Mack's. The `total=False` path reports a different (Mack 1997) probability test.
- No **weighted residuals** (Mack Appendix G / Appendix H plots) by calendar year, development age or origin, so part b needs hand-built residuals and plots.
- Suggested feature: fix `z_critical`, add `ValuationCorrelation.summary_` (S, L, Z, E, Var by diagonal), and a `MackResiduals` estimator (below).

## Sketch: `MackResiduals`

Prototype (not part of chainladder) returning the weighted residuals in long form so they can be grouped or plotted by calendar year, origin or maturity.

In [4]:
from sklearn.base import BaseEstimator


class MackResiduals(BaseEstimator):
    """Weighted residuals (f_ik - f_k) * sqrt(C_ik), optionally divided by sigma_k."""

    def __init__(self, standardize=False):
        self.standardize = standardize

    def fit(self, X, y=None):
        C = X.values[0, 0]
        origins = [str(o)[:4] for o in X.origin.astype(str)]
        recs = []
        for k in range(C.shape[1] - 1):
            rows_k = np.where(~np.isnan(C[:, k + 1]))[0]
            fk = C[rows_k, k + 1].sum() / C[rows_k, k].sum()
            fi = C[rows_k, k + 1] / C[rows_k, k]
            s = np.sqrt((C[rows_k, k] * (fi - fk) ** 2).sum() / (len(rows_k) - 1)) if len(rows_k) > 1 else np.nan
            for i, r in zip(rows_k, fi):
                w = (r - fk) * np.sqrt(C[i, k])
                recs.append({"origin": int(origins[i]), "maturity": int(X.ddims[k + 1]),
                             "calendar_year": int(origins[i]) + k + 1,
                             "residual": w / s if self.standardize else w})
        self.residuals_ = pd.DataFrame(recs)
        return self


mr = MackResiduals().fit(tri)
print(mr.residuals_.query("maturity <= 48").pivot_table(index="calendar_year", columns="maturity", values="residual").round(1))
# mr.residuals_.plot.scatter("calendar_year", "residual")   # reproduces the graph in the problem
assert np.isclose(mr.residuals_.query("calendar_year == 2016 and maturity == 24")["residual"].iloc[0], resid.query("CY == 2016 and maturity == 24")["residual"].iloc[0])

maturity         24    36   48
calendar_year                 
2014            6.5   NaN  NaN
2015            6.8   2.6  NaN
2016           26.2  12.5  7.1
2017          -16.2  -6.7 -3.1
2018          -12.5  -6.4 -3.5


Extensions not prototyped: a `.plot()` method (residuals by calendar year, origin and maturity), a corrected `ValuationCorrelation.z_critical`, and the per-diagonal summary table.